# Group 2607 Project Presentation

## **Introduction**

### SAT problems
In this project, we worked on the application of QAOA to the classical SAT problems, focusing specifically on the 2-SAT problem while also presenting some brief results for the 3-SAT problem. 

The general k-SAT and 1-in-k-SAT problems represent some of the most used methods to test and benchmark algorithm performance. A single instance consists of a formula containing $m$ clauses with $k$ variables each. These variables are sampled randomly out of $n$ boolean variables and are called literals when used in a clause. Literals must be unique in a clause. They can be either positive or negated.
For example, a 2-SAT problem with $n=6$ and $m=3$ can be represented by the following formula:
\begin{equation*}
\mathcal{F}=[(x_1\lor \neg{x_2})\land(x_3\lor x_6)\land(\neg x_4 \lor x_1)].
\end{equation*}
As mentioned before, the variables used in a formula are sampled randomly out of the $n$ total available variables. In this specific case with $n=6$, it is clear that $x_5$ was not used. A solution to this problem is a bit string $(x_1,x_2,x_3,x_4,x_5,x_6)$ where a boolean is assigned to each variable in order to satisfy every clause in the formula.
In the case of 1-in-k-SAT problems a constraint is added, forcing the search algorithm to find a bitstring such that every clause contains exactly one True literal.

These problems are characterized by a SAT regime, where there exists at least one bitstring that satisfies all clauses, and an UNSAT regime, where no solutions exist. The phase transition between the SAT and UNSAT regime is encountered at specific values of the critical density $\alpha=m/n$, where $m$ is the number of clauses and $n$ is the number of variables. The characteristic critical densities for the 2-SAT and 3-SAT problems are respectively $\alpha_c=1$ and $\alpha_c=4.26$, as discussed in the paper *"Analytic and Algorithmic Solution of Random Satisfiability Problems"* by M. Mezard, G. Parisi, and R. Zecchina.

### Quantum computing approach(QAOA)
In our work the main focus was to study how quantum computing, specifically the Quantum Approximate Optimization Algorithm (QAOA), can be used to tackle these problems, analyzing whether it provides an improvement over the classical approach.
QAOA is a hybrid quantum-classical algorithm designed to solve optimization problems. It makes use of a quantum circuit and a classical optimizer. First, the objective function is mapped into a quantum cost Hamiltonian $H_C$. The algorithm then employs a parametrized quantum circuit of $p$ layers to be optimized. The qubits are initialized in a state $\ket{\psi_0}=\ket{+}^{\otimes{n}}$, which then evolves through $p$ layers, each of which depends on two parameters $(\gamma_i,\beta_i)$.

Each layer applies two unitary evolutions, one generated by the cost Hamiltonian $H_C$ and the other one by a mixing Hamiltonian $H_B$. This results in constructive interference which increases the probability amplitudes of the low-cost solutions.

The final state is then measured multiple times to evaluate the expectation $\langle H_C \rangle$. Finally, the circuit(i.e. the $2p$ parameters ($\vec{\gamma}$,$\vec{\beta}$)) is optimized with a classical optimizer to find the best parameters. The circuit is then run with the new parameters. This procedure is repeated in loop until convergence.

## **Formula Generation and Classical Solver**

## Formula Generation
To generate formulas we took an object oriented approach, creating the class `kSATGenerator` that takes as parameters the number of variables `k` in each clause and eventually a seed to be specified. To generate the formula the function `generate` has to be called and initialized with the number of caluses `m` and the number of variables `n`. The maximum number of clauses is calculated to introduce a security check over the specified `m` value. Every clause is stored in the `clauses` set that it is converted in a list at the end. A single clause is generated by sampling radnomly $k$ out of the total $n$. Each of these variables is then turned into a literal by multiplying it randomly with plus or minus one, to indicate its positive or negated state. The example formula presented in the introduction, if generated with this approach, would look like this:
\begin{equation*}
\mathcal{F}=[(1, -2), (3, 6), (-4, 6)].
\end{equation*}

In [ ]:
class kSATGenerator:
    def __init__(self, k, seed=None):
        self.k = k
        self.random = random.Random(seed)

    def generate(self, num_clauses, num_vars):
        max_num_clauses = 2**self.k * math.comb(num_vars, self.k)
        if num_clauses > max_num_clauses:
            raise ValueError("Too many clauses")
        
        vars_list = list(range(1, num_vars + 1))
        clauses = set()

        while len(clauses) < num_clauses:
            variables = sorted(self.random.sample(vars_list, self.k))
            literals = [var * self.random.choice([1, -1]) for var in variables]
            clause = tuple(literals)
            clauses.add(clause)

        return list(clauses)

### Approaching the problem "classically"
The `brute_force_solve` function takes as an input a formula generated by the `kSATGenerator` function described in the previous section, and implements a brute force approach to look for solutions. Given the number of variables $n$, it tests all the $2^n$ possible bitstrings of length $n$ against the formula. 

A boolean parameter named `one_in_k` can be toggled to solve the 1-in-k-SAT variant, restricting the search to bitstrings that yield exactly one True literal per clause. 

The algorithm makes use of a `for` loop in order to explore the entire solution space. Therefore it is clear that this approach is not scalable for high values of $n$, since the computational complexity scales like $\mathcal{O}(2^n)$. The loop stops when it finds the first valid solution, returning the bitstring satisfying the formula. If no solution has been found throughout the entire loop, it simply returns `None`.

In [ ]:
def brute_force_solve(formula, one_in_k=False):
    num_variables = get_num_variables(formula)
    for n in range(2**num_variables):
        violated_count = 0
        for clause in formula:
            true_count = 0
            for literal in clause:
                var_index = abs(literal) - 1
                value = (n >> var_index) & 1
                if literal < 0:
                    value = 1 - value
                    
                true_count += value
            if one_in_k:
                if true_count != 1:
                    violated_count += 1
            else:
                if true_count == 0:
                    violated_count += 1
        if violated_count == 0:
            bitstring = format(n, f"0{num_variables}b")
            return bitstring
    return None

In the UNSAT regime we need to find out how far away we are from having a solution. This is what is called the Max-SAT problem, which consists of finding the maximum number of clauses that can be satisfied for a given formula. By counting the number of violated clauses for all the possible $2^n$ bitstrings for $n$ variables, we get what is called the full spectrum of that formula.

The `exact_maxsat` function was introduced to solve the Max-SAT problem, i. e. to find the minimum of the full spectrum. Just like the `brute_force_solve`, this approach is not scalable because of the computational complexity exponentially increasing with respect to $n$.

In [ ]:
def exact_maxsat(formula, num_vars):
    N = 2**num_vars
    states = np.arange(N, dtype=np.int32)
    violated_counts = np.zeros(N, dtype=np.int32) # will store how many clauses each bitstring violates

    for clause in formula:
        clause_violated = np.ones(N, dtype=bool) # all clauses are violated
        for lit in clause:
            # the following two rows extract the value of the variable in the bitstring
            var_idx = abs(lit) - 1
            bit_values = (states >> var_idx) & 1

            if lit > 0:
                lit_false = (bit_values == 0)
            else:
                lit_false = (bit_values == 1)
            clause_violated &= lit_false # update: we 
        violated_counts += clause_violated
        
    min_violated = np.min(violated_counts)
    return len(formula) - min_violated

## **Mapping the SAT problem onto the QAOA framework**

To map the problem onto the QAOA framework, we must construct the cost Hamiltonian $H_C$. Given a formula with $n$ variables, its cost Hamiltonian will be a $2^n\times 2^n$ diagonal matrix in the computational basis $\{\ket{00\dots 0},\ket{00\dots 1}, \ket{11\dots 1}\}$. The eigenstates of this Hamiltonian correspond to the $2^n$ possible bitstrings. The energy levels, i. e. the eigenvalues on the diagonal, represent the number of clauses violated by the corresponding bitstring.

To construct this matrix from a given formula, the `k_sat_hamiltonian` function was introduced.

In [ ]:
def ksat_hamiltonian(formula):
    n_vars = get_num_variables(formula)

    dim = 2 ** n_vars
    H_diag = np.zeros(dim, dtype=int)

    for clause in formula:
        ith_term = np.ones(2 ** n_vars, dtype=int)
        for literal in clause:
            variable_idx = np.abs(literal) - 1
            
            states = np.arange(dim)
            bits = (states >> variable_idx) & 1
            if literal > 0:
                proj = 1 - bits
            else:
                proj = bits
            ith_term *= proj
        H_diag += ith_term
    H = diags(H_diag, 0, format='csr')
    return H

In order to run QAOA, we need to construct the cost Hamiltonian $H_C$ in Qiskit. To do so, we need to write it in terms of the identity matrix $\mathbb{1}$ and of the Pauli matrices $\sigma^x, \sigma^y, \sigma^z$.

The $k$-SAT problem is encoded in the following formula for the Hamiltonian:
$$
\begin{equation*}
    H_C = \frac{1}{2^k}\sum_{a=1}^m\prod_{l=1}^k(\mathbb{1}+A_{a_l,a}\sigma_{a_l}^z),
\end{equation*}
$$
where
* $m$ is the number of variables,
* $k$ is the number of literals per clause,
* $a_l$ is the index of the specific variable($l$-th literal of the $a$-th clause),
* $\sigma_{a_l}^z$ is the Pauli-Z operator, acting on the qubit $a_l$,
* $A_{a_l,a} \in \{+1, -1\}$ is a sign coefficient indicating whether the literal is positive or negated in the formula.

In [ ]:
def sparse_pauli_list_2sat(formula):
    assert get_k(formula) == 2
    n_vars = get_num_variables(formula)
    
    coeffs = {}

    def add_term(z_vars, coeff):
        string = ['I'] * n_vars
        for x in z_vars:
            string[-x] = 'Z'
        string = "".join(string)
        coeffs[string] = coeffs.get(string, 0) + coeff

    for (lit1, lit2) in formula:
        sgn1, sgn2 = int(np.sign(lit1)), int(np.sign(lit2))
        var1, var2 = np.abs(lit1), np.abs(lit2)

        add_term([], 1)
        add_term([var1], sgn1)
        add_term([var2], sgn2)
        add_term([var1, var2], sgn1*sgn2)

    return [(pauli, coeff / 4) for pauli, coeff in coeffs.items() if coeff != 0]


def sparse_pauli_list_ksat(formula):
    k = get_k(formula)
    n_vars = get_num_variables(formula)

    coeffs = {}

    def add_term(z_vars, coeff):
        string = ['I'] * n_vars
        for q in z_vars:
            string[-q] = 'Z'
        string = "".join(string)
        coeffs[string] = coeffs.get(string, 0) + coeff

    for clause in formula:
        terms = [(1, [])]
        # Multiply by (1 + s_i Z_i)
        for lit in clause:
            sgn = int(np.sign(lit))
            var = abs(lit)
            new_terms = []
            for coeff, z_vars in terms:
                # Multiply by 1
                new_terms.append((coeff, z_vars))
                # Multiply by s_i Z_i
                new_terms.append((coeff * sgn, z_vars + [var]))
            terms = new_terms

        # Add to global coeffs
        for coeff, z_vars in terms:
            add_term(z_vars, coeff)

    return [(pauli, coeff / 2 ** k) for pauli, coeff in coeffs.items() if coeff != 0]

FEDERICO STA COMMENTANDO SPARSE PAULI

## **Quantum Circuit and QAOA Solver**

The simulated circuit is created throuugh the **QAOA_SATsolver** function.

In [ ]:
def QAOA_SATsolver(formula,
                   p,
                   optimizer = "COBYLA",
                   steps_optim = 1000,
                   initial_params = None,
                   seed = None,
                   verbose = False,
                   trial_idx = None
                   ):

Here the initialized parameters are just an example. The true parameters were always specified in our "control panel" script `multiSAT.py`. If not specified, the initial parameters are sampled uniformly out of a small range of values $[0,0.1]$.

At the core of the function lies the generation of the cost Hamiltonian $H_C$ using the Sparse Pauli list described previously. This operator is then fed to Qiskit `QAOAAnsatz` function, that generates the corresponding abstract circuit with `p` layers. It also automatically generates the default Mixing Hamiltonian $H_B=\sum_{i=1}^n\sigma_i^x$.

 This raw ansatz is then passsed to the Qiskit transpiler, that adapts it to the designated simulator used, `AerSimulator` and a single round of optimization is performed to simplify the circuit deleting possible redundant logic gate. We chose AerSimulator given the fact it can simulate the noise that characterizes real quantum hardware, while also providing different simulation tools like `StateVector`, `Density Matrix` or `Matrix Product State`, automatically selected based on the presented task.

To study the output of the circuit we chose `EstimatorV2`. It performs matrices multiplication returning the exact expectation energy estimate. The use of *Primitive Unified Blocs* (PUB) makes it easy to pass as input a complete package containing all the information needed for a single energy calculation. The estimator can process a list of hundreds of PUBs making the preparatrion of inputs smooth and easy. The output consists of an array of energy expectation values for each PUB, that the optimizer will then minimize.

The target to minimize is defined in the function `cost_function`, that takes the parameters, creates the corresponding PUB and uses the esitmator to evaluate the mean energy of the circuit. At the end of the optimization the optimal parameters are assigned to the circuit. We also implemented a method to compute the run time and a dictionary to store all the important quantities calculated, such as `p`, `m`, the energy $H_C$, time of execution and optimal parameters. This will be very useful in the simulation over different `p` and `m` values

### Euristic Parameter Initialization

Starting the parameter optimization from random initial conditions for every circuit, especially for the ones with high `p`, is a lot of heavy work considering that the corresponding parameter space has dimension $2p$. To avoid it, we implemented an euristic initialization of paramters based on a layer-wise training approach. The idea consists in using the best parameters obtained through the opitimization of a circuit with `p` layers, as the starting point for the optimization of a new circuit with `p+1` layers. 

This helps prevent the optimization algorithm from getting stuck in a local minimum or **Barren Plateau** when working with circuits characterized by higher number of layers.

In [ ]:
def expand_qaoa_params(old_params, p_old, p_new):
    """
    Expands the optimal parameters from an arbitrary depth p_old to p_new.
    Respects Qiskit's parameter ordering: [beta_0..beta_p, gamma_0..gamma_p].
    """
    delta_p = p_new - p_old
    
    # Se per qualche motivo p_new non è maggiore, non facciamo nulla
    if delta_p <= 0:
        return old_params
        
    new_params = np.zeros(2 * p_new)
    
    # Extract old betas and gammas
    old_betas = old_params[:p_old]
    old_gammas = old_params[p_old:]
    
    # Generate delta_p new random parameters (small noise) for the new layers
    new_betas_padding = np.random.uniform(-0.01, 0.01, size=delta_p)
    new_gammas_padding = np.random.uniform(-0.01, 0.01, size=delta_p)
    
    # Append the new layers to the old ones
    new_betas = np.append(old_betas, new_betas_padding)
    new_gammas = np.append(old_gammas, new_gammas_padding)
    
    # Reassemble in Qiskit's required order
    new_params[:p_new] = new_betas
    new_params[p_new:] = new_gammas
    
    return new_params

This solution is implemented in the `expand_qaoa_params` function. First it checks if the old circuits has indeed a lower number of layers than the new one, then extracts its best $\vec{\gamma}$ and $\vec{\beta}$ vectors. A padding layer is added to the new parameters lists to avoid the presence of parameters equal to zero. The new list is generated at the end using the format required by Qiskit $[\beta_0,\dots,\beta_p,\gamma_0,\dots,\gamma_p]$ and then used to initilize the depper new circuit.

### Simulation Engine

In [ ]:
def gridsearch_QAOA_SATsolver(
        k,
        num_vars,
        p_values, 
        m_values, 
        n_trials=50,
        trial_start=0,
        optimizer="L-BFGS-B",
        steps_optim=10000,
        n_jobs=-1, 
        verbose=False,
        dir_name="results",
        run_name="default_run"):

The real simulation mechanics are defined in the `gridsearch_QAOA_SATsolver` function, that takes all the crucial parameters to use in a run. The first thing this function does is creating a dictionary to store the metadata of the specific run initialized with those parameters.

The real engine is the `worker` function. It defines the tasks each CPU core has to complete. To ensure complete reproducibility, even when faced with disconnection or crashes of the virtual machines used, a deterministic seed was implemented performing hashing on every combination of value $(m,trial\ index)$, where the trial index indicates a certain generated formula for the problem. This helped us divide the workload between different computers by assigning to each one a certain number of formulas generated with this specific seed.

In [ ]:
 def worker(m, trial_idx):
        # create a seed for this specific run
        seed_string = f"m{m}_t{trial_idx}"
        trial_seed = int(hashlib.md5(seed_string.encode()).hexdigest(), 16) % (2**32)
        gen = kSATGenerator(k=k, seed=trial_seed)
        formula = gen.generate(num_clauses=m, num_vars=num_vars)
        c_max = exact_maxsat(formula, num_vars) 
        
        results_for_this_formula = []
        current_initial_params = None
        current_p = 0
        
        
        # ensure p_values are sorted for sequential layerwise training
        for p in sorted(p_values):
            if current_initial_params is None:
                rng = np.random.default_rng(trial_seed)
                current_initial_params = rng.uniform(0, 0.1, size=2*p)
            else:
                current_initial_params = expand_qaoa_params(current_initial_params, current_p, p)
        

            result = QAOA_SATsolver(
                formula=formula,
                p=p,
                optimizer=optimizer,
                steps_optim=steps_optim,
                initial_params=current_initial_params,
                seed=trial_seed,
                verbose=verbose,
                trial_idx=trial_idx
            )
            
            # Save the optimized parameters to initialize the next p iteration
            current_initial_params = np.array(result["opt_params"])
            current_p = p
            
            del result["opt_params"]

            # add metadata for tracking
            result["trial"] = trial_idx
            result["C_max"] = c_max
            result["C_qaoa"] = m - result["energy_Hc"]
            result["approx_ratio"] = result["C_qaoa"] / c_max if c_max > 0 else 1.0 

            results_for_this_formula.append(result)
            
        return results_for_this_formula

A classic baseline $C_{max}$ is calculated using the `exact_maxsat` function, which will be the reference to evaluate the performance of the circuit through the approximation ratio $r$. Every worker sorts the list of `p` layers specified and starts iteratively assigning the initial $(\vec{\beta},\vec{\gamma})$: the circuit with the lowest `p` is initialized with random near zero parameters, while the other ones will be initilized using the euristic initializaiton described before.
 `QAOA_SATsolver` is called for each `p` and its output is saved to be used as starting point for the next `p` iteration. 

The number of satisfied clauses $C_{qaoa}$ is calculated and used to evaluate the ciruit performance, $r=C_{qaoa}/C_{max}$. All these information and other important metadata, like the index of the formula studied in one iteration, are added as new columns to the solver output.

The last part of the function generates the parallelization framework, using `joblib.Parrallel`. First a set of instances $(m,trial)$ is created. These instances will be sent to every CPU core to be analyzed in parallel. Every core though, will process a single istance iteratively for every `p` to ensure the euristic initialization.

At the end all the simulation data will be stored in a `.csv` file with the chosen `run_name` and a metadata `.json` file will be generated.  

## **Optimizers**

DA SCRIVERE POMERIGGIO

## **QAOA Results**

PLOTS DA INSERIRE POMERIGGIO

## **Gradients**

Since QAOA is a variational quantum algorithm, the trainability of the circuit is related to the behavior of the gradients of the cost function with respect to the variational parameters $\vec{\gamma}$ and $\vec{\beta}$.
Large gradient magnitudes correspond to a rapidly varying cost-function landscape, which facilitates optimization.
In contrast, small gradients are associated with barren plateaus that is regions where the cost function is flat and optimization more difficult.
For a given formula, the cost function is implemented in python as follows:

In [ ]:
sparse_pauli_list = sparse_pauli_list_ksat(formula)
cost_hamiltonian = SparsePauliOp.from_list(sparse_pauli_list)
        
raw_ansatz = QAOAAnsatz(cost_hamiltonian, reps=p)
ansatz = transpile(raw_ansatz, backend=backend, optimization_level=1)

def cost_function(params):
    pub = (ansatz, cost_hamiltonian, params)
    job = estimator.run([pub])
    result = job.result()[0]
    return result.data.evs

Because gradients average to zero over random parameter configurations, we characterize their typical magnitude using the standard deviation (SD) of the gradients magnitude rather than their mean.
Gradients are evaluated at 5000 randomly sampled circuit parameters in $[0,2\pi]^{2p}$ using a numerical finite-difference method implemented in scipy.
For describing the cost-function landscape as a function of m/n, it is reasonable to expect that the partial derivative of the gradient w.r.t. one coordinate can as well be used instead of the gradient magnitude.
To compute the gradient magnitude at a random point:

In [ ]:
params = np.random.uniform(low=0, high=2*np.pi, size=ansatz.num_parameters)
grad = approx_fprime(params, cost_function)

Instead, to compute the partial derivative w.r.t., wlog, $\gamma_1$:

In [ ]:
params = np.random.uniform(low=0, high=2*np.pi, size=ansatz.num_parameters)

def f1(x):
    p_copy = params.copy()
    p_copy[0] = x[0]
    return cost_function(p_copy)

partial = approx_fprime(np.array([params[0]]), f1)

We compute the gradient magnitude/partial derivative SD on 5000 points, and average it over some iterations (50 or 100, see later).
For visualization purposes, we plot the average of the inverse SD.

PLOTS...

Larger values indicate smaller typical gradients and therefore more difficult optimization.
We see that, when the QAOA depth $p$ is small, this peak is no visible; however, in this regime QAOA also fails to produce accurate solutions, making trainability irrelevant.
For large values of $p$, peaks are clearly visible.
For 2-SAT, the peak is at a clause-to-variable ratio larger than the SAT-UNSAT phase transition ratio.
For 3-SAT, the peak is at a smaller clause-to-variable ratio than the SAT-UNSAT transition ratio.
Recall that the SAT-UNSAT phase transition ratio corresponds to the region of maximum empirical hardness for classical algorithms.
Thus we see that the empirical hardness for QAOA is different from classical algorithms.
Although this result holds for SAT problem, we expect the __computational__ phase transition in QAOA to apply to all
combinatorial optimization problems. In particular, as 3-SAT is NP-complete, the clause-to-variable ratio represents a universal characterization of a ‘problem density’, and the computational phase transition applies to all NP-complete problems in this regard. 